# 06 — Filtering YouTube Videos

**Pipeline step:** Sections 3.1.2–3.1.3 of the paper — domain-based filtering of the raw URL
dataset (justifying Table 1) and extraction of the YouTube subset.

**Purpose.**
1. Inspect raw domains to justify filtering down to major social platforms (Table 1).
2. Count links per platform, matching the domain patterns reported in the paper's Table 1.
3. Extract all YouTube URLs into their own file (`youtube_links.jsonl`).
4. Reproduce the YouTube-only version of the monthly occurrence plot
   (`contagem_ocorrencias_yt.png`).

**Input:**
- `../data/telegram_2024/urls.json.gz` — all extracted URLs (line-delimited JSON), same source
  used in `05_Links.ipynb`.

**Output:**
- `../data/telegram_2024/youtube_links.jsonl` — YouTube-only subset of the URL dataset.
- `../reports/Figures/contagem_ocorrencias_yt.png`.

**Next step:** `07_Extracting_infos_yt.ipynb`, which enriches each video via the YouTube Data API.

> ⚠️ **Documentation gap found here — needs your input.** Section 3.1.3 of the paper reports a
> breakdown of YouTube URLs by content type (video: 1,602,384; short: 161,247; live: 137,615;
> playlist: 5,686; channel: 4,498; clip: 907; other: 8,455). In the original notebook these
> numbers appeared as a hand-typed markdown note, with no code that computes them. I did not
> reconstruct this logic myself — a URL-classification regex I guessed might not reproduce
> these exact published numbers, which would silently make the paper's own reported figures
> non-reproducible. See the flagged cell near the bottom of this notebook: please paste in
> (or point me to) the original classification code so I can turn it into real, tested code
> instead of a static comment.


In [ ]:
import gzip
import json
from collections import Counter
from pathlib import Path
from urllib.parse import urlparse

import matplotlib.pyplot as plt
import pandas as pd
from tqdm import tqdm

In [ ]:
URLS_JSONL_GZ_PATH = "../data/telegram_2024/urls.json.gz"
YOUTUBE_LINKS_PATH = "../data/telegram_2024/youtube_links.jsonl"
FIGURES_DIR = Path("../reports/Figures")

# Domain patterns per platform — matches Table 1 in the paper.
PLATFORM_DOMAINS = {
    "x": ["twitter.com", "x.com", "t.co"],
    "youtube": ["youtube.com", "youtu.be"],
    "instagram": ["instagram.com"],
    "rumble": ["rumble.com"],
    "tiktok": ["tiktok.com"],
    "facebook": ["facebook.com", "fb.com", "fb.watch"],
    "vk": ["vk.com", "vkontakte.ru"],
    "truthsocial": ["truthsocial.com"],
}

## 1. Sanity check: total raw URL records

In [ ]:
with gzip.open(URLS_JSONL_GZ_PATH, "rt") as f:
    total_records = sum(1 for _ in f)

print(f"Total raw URL records: {total_records}")

## 2. Raw domain exploration

Before applying the platform filter, we look at *all* domains present in the raw data — this
manual inspection is what motivated restricting the analysis to major social platforms
(Section 3.1.2), since many domains fall outside the scope of the study (crypto, NFTs,
promotional pages, etc.).

In [ ]:
domain_counts = Counter()
n_malformed = 0

with gzip.open(URLS_JSONL_GZ_PATH, "rt", encoding="utf-8") as f:
    for line in tqdm(f, desc="Extracting domains", total=total_records):
        if not line.strip():
            continue
        try:
            item = json.loads(line)
        except json.JSONDecodeError:
            continue

        full_url = item.get("url")
        if not full_url:
            continue

        # A small fraction of shared URLs are malformed (stray characters, unbalanced
        # punctuation copied alongside the link, full-width punctuation, etc.) and make
        # urlparse raise a ValueError. We skip those and keep a count for transparency.
        try:
            domain = urlparse(full_url.lower()).netloc
        except ValueError:
            n_malformed += 1
            continue

        if domain.startswith("www."):
            domain = domain[4:]
        if domain:
            domain_counts[domain] += 1

print(f"Skipped {n_malformed} malformed URLs ({n_malformed / total_records:.4%} of the total).")
print("\nTop 30 domains overall:")
for domain, count in domain_counts.most_common(30):
    print(f"{domain:<30} | {count} occurrences")

## 3. Link counts per platform (Table 1)

In [ ]:
platform_link_counts = {platform: 0 for platform in PLATFORM_DOMAINS}

with gzip.open(URLS_JSONL_GZ_PATH, "rt") as f:
    for line in tqdm(f, desc="Classifying links by platform", total=total_records):
        try:
            data = json.loads(line)
        except json.JSONDecodeError:
            continue
        url = data.get("url", "").lower()

        for platform, domains in PLATFORM_DOMAINS.items():
            if any(domain in url for domain in domains):
                platform_link_counts[platform] += 1
                break  # a URL is attributed to a single platform

print("Links per platform:")
for platform, count in platform_link_counts.items():
    print(f"{platform}: {count}")

print(f"\nTotal links across tracked platforms: {sum(platform_link_counts.values())}")

## 4. Extracting YouTube links

Writes every record whose URL matches a YouTube domain to its own file, so downstream
notebooks don't need to re-scan the full multi-platform dataset.

In [ ]:
youtube_domains = PLATFORM_DOMAINS["youtube"]
n_extracted = 0

with gzip.open(URLS_JSONL_GZ_PATH, "rt") as f, open(YOUTUBE_LINKS_PATH, "w") as out:
    for line in tqdm(f, desc="Extracting YouTube links", total=total_records):
        try:
            data = json.loads(line)
        except json.JSONDecodeError:
            continue
        url = data.get("url", "").lower()
        if any(domain in url for domain in youtube_domains):
            out.write(json.dumps(data) + "\n")
            n_extracted += 1

print(f"Extracted {n_extracted} YouTube links to {YOUTUBE_LINKS_PATH}")

## 5. YouTube content-type breakdown (Section 3.1.3)

> ⚠️ **TODO — see the callout at the top of this notebook.** The paper reports a breakdown
> of `youtube_links.jsonl` by content type (video / short / live / playlist / channel / clip /
> other), used to justify keeping only videos, Shorts, and live streams downstream. That
> classification logic is missing here — please supply it (e.g. the URL-path rules used to
> tell a `/watch?v=` video apart from a `/shorts/`, a channel page, a playlist, etc.) so this
> section can compute the counts from `youtube_links.jsonl` instead of restating them as text.

In [ ]:
# TODO: compute content-type counts from YOUTUBE_LINKS_PATH once the classification
# rules are provided. Expected categories (from the paper): video, short, live, playlist,
# channel, clip, other.

## 6. Loading extracted YouTube links

In [ ]:
records = []
with open(YOUTUBE_LINKS_PATH, "r", encoding="utf-8") as f:
    for line in f:
        if line.strip():
            records.append(json.loads(line))

# One row per (url, occurrence) pair: each occurrence is a single posting of that URL
# in a given chat ("folder") at a given time ("date").
occurrences = pd.json_normalize(records, record_path="occurrences", meta=["url"])
occurrences.head()

## 7. Diagnostic: first posting vs. reposts over time

For each URL, the first time it appears in a given chat is plotted in red; every later
repost of the same URL in the same chat is plotted in black. This is a quick visual check
of how repostings cluster over time.

In [ ]:
occurrences["date"] = pd.to_datetime(occurrences["date"])
occurrences = occurrences.sort_values("date")

unique_urls = occurrences["url"].unique()
url_id_map = dict(zip(unique_urls, range(len(unique_urls))))
occurrences["url_id"] = occurrences["url"].map(url_id_map)
occurrences["is_repost"] = occurrences.duplicated(subset=["folder", "url_id"])

fig, ax = plt.subplots(figsize=(12, 7))
reposts = occurrences[occurrences["is_repost"]]
first_posts = occurrences[~occurrences["is_repost"]]

ax.scatter(reposts["date"], reposts["url_id"], marker=".", s=0.1, alpha=0.1, c="k", label="Repost")
ax.scatter(first_posts["date"], first_posts["url_id"], marker=".", s=0.1, alpha=0.1, c="r", label="First post")
ax.set_xlabel("Date")
ax.set_ylabel("URL id")
plt.tight_layout()
plt.show()

## 8. URL occurrences per month — YouTube subset

In [ ]:
monthly_counts = Counter()

with open(YOUTUBE_LINKS_PATH, "rt", encoding="utf-8") as f:
    for line in tqdm(f, desc="Counting monthly occurrences"):
        if not line.strip():
            continue
        try:
            item = json.loads(line)
        except json.JSONDecodeError:
            continue
        for occurrence in item.get("occurrences", []):
            if "date" in occurrence:
                month = occurrence["date"][:7]  # YYYY-MM
                monthly_counts[month] += 1

In [ ]:
months_ordered = sorted(monthly_counts.keys())
values_ordered = [monthly_counts[m] for m in months_ordered]

plt.figure(figsize=(12, 7))
plt.bar(months_ordered, values_ordered, color="skyblue")
plt.title("Number of YouTube URL occurrences per month-year", fontsize=16)
plt.ylabel("Occurrences", fontsize=12)
plt.xlabel("Month-Year", fontsize=12)
plt.xticks(rotation=45, ha="right")
plt.grid(axis="y", linestyle="--", alpha=0.7)
plt.tight_layout()

FIGURES_DIR.mkdir(parents=True, exist_ok=True)
plt.savefig(FIGURES_DIR / "contagem_ocorrencias_yt.png")
plt.show()